In [ ]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid
from datetime import date

task_id = 50
parent_run_id = ""
task_executions_id = ""
lineage_id = str(uuid.uuid4())
source_connection_settings = "{}"
limit_rows_for_debugging = False

# Manual fallback for source settings.
source_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "folder_path": "Files/csv_data",
    "source_system": "lh_canada_global_mds",
    "table_config": [
        {"source_file": "EFI.csv", "target_table": "silver_transactional_efi"},
        {"source_file": "TME.csv", "target_table": "silver_transactional_tme"},
        {"source_file": "ChronoMetrics.csv", "target_table": "silver_transactional_chrono_metrics"}
    ]
})

# Manual fallback for target settings.
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_transactional",
    "load_strategy": "overwrite"
})

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 15, Finished, Available, Finished, False)

In [ ]:
%run nb_transactional_shared_functions

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 20, Finished, Available, Finished, True)

In [ ]:
# Standard framework logging setup.
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 21, Finished, Available, Finished, False)

In [ ]:
# Accept either JSON strings (framework) or already-parsed dicts (manual/debug).
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_lakehouse_name = source_settings.get("lakehouse_name")
source_folder_path = source_settings.get("folder_path")
source_system = source_settings.get("source_system", source_lakehouse_name)
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "overwrite")

# Resolve lakehouse storage paths by name.
source_lakehouse_path = notebookutils.lakehouse.get(source_lakehouse_name).properties["abfsPath"]
target_lakehouse_path = notebookutils.lakehouse.get(target_lakehouse_name).properties["abfsPath"]

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 22, Finished, Available, Finished, False)

In [ ]:
# Load csv to delta
status = "Failed"

try:
    for cfg in table_config:
        source_file = cfg["source_file"]
        target_table = cfg["target_table"]

        # Build source file path and target delta table path.
        source_file_path = f"{source_lakehouse_path}/{source_folder_path}/{source_file}"
        target_table_path = f"{target_lakehouse_path}/Tables/{target_schema}/{target_table}"

        logger.info(f"Processing {source_file} -> {target_schema}.{target_table}")
        logger.info(f"Source path: {source_file_path}")
        logger.info(f"Target path: {target_table_path}")

        # Read source CSV file.
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("delimiter", ",")
            .load(source_file_path)
        )

        # Apply framework-aligned cleanup.
        df = clean_bronze_table(df, limit_rows_for_debugging)

        # Remove exact duplicate rows.
        before_count = df.count()
        df = remove_exact_duplicates(df)
        after_count = df.count()

        logger.info(f"{source_file} removed {before_count - after_count} duplicate rows")

        # Stamp framework metadata columns.
        df = add_metadata(
            df=df,
            ingest_date=str(date.today()),
            file_path=source_file_path,
            schema_name=source_system,
            table_name=target_table,
            lineage_id=lineage_id
        )

        # Write to the target table path.
        # For this base notebook we only care about overwrite/append.
        load_to_target(df, target_table_path, target_load_strategy)

        logger.info(f"Loaded {source_file} into {target_table_path}")

    status = "Completed"

except Exception:
    logger.exception("Processing error.")
    status = "Failed"
    raise

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 23, Finished, Available, Finished, False)

2026-06-26 21:39:46,163 UTC - INFO - Processing EFI.csv -> silver_transactional.silver_transactional_efi (LoadTransactionalToBase)
2026-06-26 21:39:46,163 UTC - INFO - Source path: abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Files/csv_data/EFI.csv (LoadTransactionalToBase)
2026-06-26 21:39:46,165 UTC - INFO - Target path: abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_efi (LoadTransactionalToBase)


2026-06-26 21:39:48,207 UTC - INFO - EFI.csv removed 0 duplicate rows (LoadTransactionalToBase)
2026-06-26 21:39:48,295 UTC - INFO - Overwriting records in abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_efi (LoadTransactionalToBase)


2026-06-26 21:39:51,205 UTC - INFO - Loaded EFI.csv into abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_efi (LoadTransactionalToBase)
2026-06-26 21:39:51,206 UTC - INFO - Processing TME.csv -> silver_transactional.silver_transactional_tme (LoadTransactionalToBase)
2026-06-26 21:39:51,206 UTC - INFO - Source path: abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Files/csv_data/TME.csv (LoadTransactionalToBase)
2026-06-26 21:39:51,207 UTC - INFO - Target path: abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_tme (LoadTransactionalToBase)


2026-06-26 21:39:53,654 UTC - INFO - TME.csv removed 0 duplicate rows (LoadTransactionalToBase)
2026-06-26 21:39:54,146 UTC - INFO - Writing new table to abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_tme (LoadTransactionalToBase)


2026-06-26 21:40:25,524 UTC - INFO - Loaded TME.csv into abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_tme (LoadTransactionalToBase)
2026-06-26 21:40:25,525 UTC - INFO - Processing ChronoMetrics.csv -> silver_transactional.silver_transactional_chrono_metrics (LoadTransactionalToBase)
2026-06-26 21:40:25,525 UTC - INFO - Source path: abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Files/csv_data/ChronoMetrics.csv (LoadTransactionalToBase)
2026-06-26 21:40:25,526 UTC - INFO - Target path: abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_chrono_metrics (LoadTransactionalToBase)


2026-06-26 21:40:29,203 UTC - INFO - ChronoMetrics.csv removed 0 duplicate rows (LoadTransactionalToBase)
2026-06-26 21:40:29,629 UTC - INFO - Writing new table to abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_chrono_metrics (LoadTransactionalToBase)


2026-06-26 21:40:31,821 UTC - INFO - Loaded ChronoMetrics.csv into abfss://d940d8ab-6c2e-4088-85b8-3335f3287cbc@onelake.dfs.fabric.microsoft.com/87edacb2-3f66-484c-929b-e69785533c31/Tables/silver_transactional/silver_transactional_chrono_metrics (LoadTransactionalToBase)


In [ ]:
# Read back from the target paths
for cfg in table_config:
    target_table = cfg["target_table"]
    target_table_path = f"{target_lakehouse_path}/Tables/{target_schema}/{target_table}"

    df_check = spark.read.format("delta").load(target_table_path)
    print(f"{target_schema}.{target_table}: {df_check.count()} rows")
    display(df_check.limit(5))

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 24, Finished, Available, Finished, False)

silver_transactional.silver_transactional_efi: 688 rows


SynapseWidget(Synapse.DataFrame, 74cfd41a-196f-40bb-95b7-0035ac78ac46)

silver_transactional.silver_transactional_tme: 3000 rows


SynapseWidget(Synapse.DataFrame, ac1a8629-873b-4c0f-a0e3-5f372f12684b)

silver_transactional.silver_transactional_chrono_metrics: 5947 rows


SynapseWidget(Synapse.DataFrame, f6d4b269-92ef-48bd-a9f4-257871ddbf29)

In [ ]:
# Task validation
result = {
    "status": status,
    "task_id": task_id,
    "task_executions_id": task_executions_id,
    "lineage_id": lineage_id,
    "source_lakehouse_name": source_lakehouse_name,
    "target_lakehouse_name": target_lakehouse_name,
    "target_schema": target_schema,
    "tables_processed": [cfg["target_table"] for cfg in table_config]
}

print(result)

StatementMeta(, 190b76b2-7c3c-4090-9ae0-314a8f95efbb, 25, Finished, Available, Finished, False)

{'status': 'Completed', 'task_id': 50, 'task_executions_id': '', 'lineage_id': '64c93357-4f5a-4efe-9aa6-0d52e7734914', 'source_lakehouse_name': 'lh_canada_global_mds', 'target_lakehouse_name': 'lh_canada_global_mds', 'target_schema': 'silver_transactional', 'tables_processed': ['silver_transactional_efi', 'silver_transactional_tme', 'silver_transactional_chrono_metrics']}
